# 可选实验：模型表示

<figure>
 <img src="./images/C1_W1_L3_S1_Lecture_b.png"   style="width:600px;height:200px;">
</figure>

## Goals
In this lab 你将：
- Learn to implement the model $f_{w,b}$ for linear regression with one variable

## Notation
Here is a summary of some of the notation you will encounter.  

|General <img width=70/> <br />  Notation  <img width=70/> | Description<img width=350/>| Python (if applicable) |
|: ------------|: ------------------------------------------------------------||
| $a$ | scalar, non bold                                                      ||
| $\mathbf{a}$ | vector, bold                                                      ||
| **Regression** |         |    |     |
|  $\mathbf{x}$ | Training Example feature values (in this lab - Size (1000 sqft))  | `x_train` |   
|  $\mathbf{y}$  | Training Example  targets (in this lab Price (1000s of dollars)).  | `y_train` 
|  $x^{(i)}$, $y^{(i)}$ | $i_{th}$Training Example | `x_i`, `y_i`|
| m | Number of 训练样本 | `m`|
|  $w$  |  parameter: weight,                                 | `w`    |
|  $b$           |  parameter: bias                                           | `b`    |     
| $f_{w,b}(x^{(i)})$ | The result of the model evaluation at $x^{(i)}$ parameterized by $w,b$: $f_{w,b}(x^{(i)}) = wx^{(i)}+b$  | `f_wb` | 


## Tools
In this lab you will make use of: 
- NumPy, a popular library for scientific computing
- Mat绘制lib, a popular library for 绘制ting data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')

# 问题陈述
<img align="left" src="./images/C1_W1_L3_S1_trainingdata.png"    style=" width:380px; padding: 10px;  " /> 

和课程中一样，你将使用房价预测这一启发性示例。
本实验将使用一个只有两个数据点的简单数据集——一套 1000 平方英尺（sqft）的房屋售价为 \\$300,000，另一套 2000 平方英尺的房屋售价为 \\$500,000。这两个点构成我们的*数据集或训练集*。在本实验中，面积单位为 1000 平方英尺，价格单位为千美元。

| 面积（1000 平方英尺）     | 价格（千美元） |
| -------------------| ------------------------ |
| 1.0               | 300                      |
| 2.0               | 500                      |

你希望用一个线性回归模型（上图中的蓝色直线）拟合这两个点，然后便可预测其他房屋的价格——例如一套 1200 平方英尺的房屋。

请运行下面的代码单元格，创建变量 `x_train` 和 `y_train`。数据存储在一维 NumPy 数组中。

In [ ]:
# x_train is the input variable (size in 1000 square feet)
# y_train is the target (price in 1000s of dollars)
x_train = np.array([1.0, 2.0])
y_train = np.array([300.0, 500.0])
print(f"x_train = {x_train}")
print(f"y_train = {y_train}")

>**注意**：本课程在打印输出时会经常使用[此处](https://docs.python.org/3/tutorial/inputoutput.html)介绍的 Python “f-string”输出格式。生成输出时，花括号中的内容会被求值。

### Number of 训练样本 `m`
You will use `m` to denote the number of 训练样本. Numpy arrays have a `.shape` parameter. `x_train.shape` returns a python tuple with an entry for each dimension. `x_train.shape[0]` is the length of the array and number of examples as shown below.

In [ ]:
# m is the number of training examples
print(f"x_train.shape: {x_train.shape}")
m = x_train.shape[0]
print(f"Number of training examples is: {m}")

也可以使用 Python 的 `len()` 函数，如下所示。

In [ ]:
# m is the number of training examples
m = len(x_train)
print(f"Number of training examples is: {m}")

### 训练样本 `x_i, y_i`

你将使用 (x$^{(i)}$, y$^{(i)}$) 表示第 $i^{th}$ 个训练样本。由于 Python 从零开始索引，(x$^{(0)}$, y$^{(0)}$) 是 (1.0, 300.0)，而 (x$^{(1)}$, y$^{(1)}$) 是 (2.0, 500.0)。

要访问 NumPy 数组中的值，可以使用所需的偏移量对数组进行索引。例如，访问 `x_train` 中位置 0 的语法是 `x_train[0]`。
运行下面的下一个代码块，获取第 $i^{th}$ 个训练样本。

In [ ]:
i = 0 # Change this to 1 to see (x^1, y^1)

x_i = x_train[i]
y_i = y_train[i]
print(f"(x^({i}), y^({i})) = ({x_i}, {y_i})")

### 绘制数据

You can 绘制 these two points using the `scatter()` function in the `mat绘制lib` library, as shown in the cell below. 
- The function arguments `marker` and `c` show the points as red crosses (the default is blue dots).

You can use other functions in the `mat绘制lib` library to set the title and labels to display

In [ ]:
# Plot the data points
plt.scatter(x_train, y_train, marker='x', c='r')
# Set the title
plt.title("Housing Prices")
# Set the y-axis label
plt.ylabel('Price (in 1000s of dollars)')
# Set the x-axis label
plt.xlabel('Size (1000 sqft)')
plt.show()

## 模型函数

<img align="left" src="./images/C1_W1_L3_S1_model.png"     style=" width:380px; padding: 10px; " > 正如课程中所述，线性回归的模型函数（即从 `x` 映射到 `y` 的函数）表示为

$$ f_{w,b}(x^{(i)}) = wx^{(i)} + b \tag{1}$$

上面的公式表示直线——$w$ 和 $b$ 取不同值时，会在图中得到不同的直线。<br/> <br/> <br/> <br/> <br/>

让我们通过下面的代码块更直观地理解这一点。先从 $w = 100$ 和 $b = 100$ 开始。

**注意：你可以返回此单元格，调整模型的 w 和 b 参数**

In [ ]:
w = 100
b = 100
print(f"w: {w}")
print(f"b: {b}")

现在, let's compute the value of $f_{w,b}(x^{(i)})$ for your two data points. You can explicitly write this out for each data point as - 

for $x^{(0)}$, `f_wb = w * x[0] + b`

for $x^{(1)}$, `f_wb = w * x[1] + b`

For a large number of data points, this can get unwieldy and repetitive. So instead, you can calculate the function output in a `for` loop as shown in the `compute_model_output` function below.
> **Note**: The argument description `(ndarray (m,))` describes a Numpy n-dimensional array of shape (m,). `(scalar)` describes an argument without dimensions, just a magnitude.  
> **Note**: `np.zero(n)` will return a one-dimensional numpy array with $n$ entries   


In [ ]:
def compute_model_output(x, w, b):
    """
    Computes the prediction of a linear model
    Args:
      x (ndarray (m,)): Data, m examples 
      w,b (scalar)    : model parameters  
    Returns
      y (ndarray (m,)): target values
    """
    m = x.shape[0]
    f_wb = np.zeros(m)
    for i in range(m):
        f_wb[i] = w * x[i] + b
        
    return f_wb

现在 let's call the `compute_model_output` function and 绘制 the output..

In [ ]:
tmp_f_wb = compute_model_output(x_train, w, b,)

# Plot our model prediction
plt.plot(x_train, tmp_f_wb, c='b',label='Our Prediction')

# Plot the data points
plt.scatter(x_train, y_train, marker='x', c='r',label='Actual Values')

# Set the title
plt.title("Housing Prices")
# Set the y-axis label
plt.ylabel('Price (in 1000s of dollars)')
# Set the x-axis label
plt.xlabel('Size (1000 sqft)')
plt.legend()
plt.show()

可以看到，设置 $w = 100$ 和 $b = 100$ 并*不会*得到一条拟合数据的直线。

### 挑战
尝试使用不同的 $w$ 和 $b$ 值。要得到一条拟合数据的直线，它们的值应该是多少？

#### 提示：
你可以用鼠标点击下方绿色“提示”左侧的三角形，以显示关于选择 b 和 w 的一些提示。

<details>
<summary>
    <font size='3', color='darkgreen'><b>提示</b></font>
</summary>
    <p>
    <ul>
        <li>尝试 $w = 200$ 和 $b = 100$ </li>
    </ul>
    </p>

### Prediction
现在 that we have a model, we can use it to make our original 预测值. Let's predict the price of a house with 1200 sqft. Since the units of $x$ are in 1000's of sqft, $x$ is 1.2.


In [ ]:
w = 200                         
b = 100    
x_i = 1.2
cost_1200sqft = w * x_i + b    

print(f"${cost_1200sqft:.0f} thousand dollars")

# 恭喜！
In this lab you have learned:
 - Linear regression builds a model which establishes a relationship between features and targets
     - In the example above, the feature was house size and the target was house price
     - for simple linear regression, the model has two parameters $w$ and $b$ whose values are 'fit' using *training data*.
     - once a model's parameters have been determined, the model can be used to make 预测值s on novel data.